# 01 — FMP Transcript Exploration

Verify transcript loading and inspect structure for KO and PEP.
Transcripts are `.txt` files delivered by Mayank into `data/raw/fmp/{TICKER}/`.
Final pipeline lives in `src/fmp_client.py`.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.fmp_client import FMPClient
from src.constants import TARGET_TICKERS

# FMPClient requires FMP_API_KEY — only needed for live API pulls.
# For local .txt transcripts, instantiate with a dummy key.
import os
os.environ.setdefault('FMP_API_KEY', 'local_only')
client = FMPClient()

In [ ]:
# Load all cached transcripts for KO and PEP
for ticker in ['KO', 'PEP']:
    transcripts = client.get_all_cached_for_ticker(ticker)
    print(f"{ticker}: {len(transcripts)} transcripts")
    for t in transcripts:
        words = len(t['content'].split())
        print(f"  {t['year']}Q{t['quarter']} | {words:,} words | source={t['source']}")

In [ ]:
# Inspect first 500 chars of a KO transcript (header should already be stripped)
ko_transcripts = client.get_all_cached_for_ticker('KO')
if ko_transcripts:
    t = ko_transcripts[0]
    print(f"{t['ticker']} {t['year']}Q{t['quarter']} ({t['filename']})")
    print(t['content'][:500])

In [ ]:
# Word count distribution across all available transcripts
import pandas as pd

rows = []
for ticker in ['KO', 'PEP']:
    for t in client.get_all_cached_for_ticker(ticker):
        rows.append({
            'ticker': t['ticker'],
            'year': t['year'],
            'quarter': t['quarter'],
            'word_count': len(t['content'].split()),
            'source': t['source'],
        })

df = pd.DataFrame(rows).sort_values(['ticker', 'year', 'quarter'])
print(df.to_string(index=False))
print(f"\nTotal transcripts: {len(df)}")
print(df.groupby('ticker')['word_count'].describe())